# ECS659U - Coursework


* The **goal** of the CW is similar to that of Week 2's Lab: fitting a curve to data, also known as **curve fitting**.
* This has applications in many different disciplines that make use of AI: FinTech, Physics Modelling, or even Sports.
* For example, we might be interested in learning the evolution (over time) of the price of a specific product in different countries. This can depend on several factors: the product itself, the country, the initial value of the product's price, etc.
* As usual, we are interested in learning a model that finds these relationships *from the data*.


## Learning a set of functions

* The main difference with Week 2's Lab that we will train a network that does not learn a single function but a set of functions.
* You are provivided with a training set ("train_data.csv") containing 30,000 functions. Each function has $100$ points (N_x = 100) on the x-axis from a $[-1, 1]$ distribution.
* Because we are dealing with a family of functions and not just a single function, our model must be able to perform two tasks: *Function Selection* and *Regression*.
* Function selection means that given some *additional* input (to be defined below) the model somehow must choose which function from the set of functions it needs to model.
* Once the correct function is picked then the model must perform regression i.e. learn the input-output relationship $y=f_a(x)$ where $f_a$ is the selected function.

In [1]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

y_train = np.loadtxt('train_data.csv', delimiter=',')
N_train, Nx = y_train.shape
x_train = np.tile(np.linspace(-1, 1, Nx), (N_train, 1))
x_tensor = torch.from_numpy(x_train).float()
y_tensor = torch.from_numpy(y_train).float()
train_loader = DataLoader(TensorDataset(x_tensor, y_tensor),batch_size=128,shuffle=True)

y_test = torch.from_numpy(np.loadtxt('test_data.csv', delimiter=',')).float()
x_test = torch.linspace(-1, 1, 100).unsqueeze(0).repeat(300, 1)

# Load the context bundle
test_contexts = torch.load('test_observations.pt')


## The Learning Objective


* For both *training and testing*, a random subset of $N$ pairs $(x_o, y_o)$ (which is called observed data) is provided. This acts as prior information to help the model identify which specific function is looking at.

* The model will take the observed pairs $(x_o, y_o)$ and target values $x_t$ and will produce the estimated values $\hat{y}_t$. The ultimate goal is to estimate the function corresponding to the curve that observed pairs $(x_o, y_o)$ belong to and then estimate the output value $\hat{y}_t$ corresponding to $x_t$. During training we have access to the ground-truth values $y_t$, and thus we can compute a loss between the model's predictions $\hat{y}_t$ and the ground-truth values $y_t$.  


## The Model Architecture

* Our model consists of 2 MLPs which must be jointly trained. Specifically, you should implement the following architecture:
* *Encoder*: The first MLP is called the Encoder. It takes as input the random set of $N$ observed pairs $(x_o, y_o)$  and maps each of them (through a series of hidden layers) to a feature vector of dimension $h_{dim}$. A final layer produces the latent feature representation $r_o$ of dimension $r_{dim}$.
* A total observed feature is produced by summing or averaging over all features (e.g., $r_O=\frac{1}{N}\sum_o r_o$).
* *Decoder*: The second MLP is called the Decoder. It takes as input the $r_O$ and each input data $x_t$. The Decoder (through a series of hidden layers) maps the pair $(r_O, x_t)$ to some feature vector of dimension $h_{dim}$. A final layer will produce the model's prediction $\hat{y}_t$ (corresponding to the input $x_t$).
* Hint! Try different choices for the number of hidden layers in the MLPs and for the hidden dimension $h_{dim}$.


## Tasks

* You have to implement the following:
    1. Create the model based on the provided architecture description. $r_{dim}$ should be a configurable hyperparameter. (20%)
    2. Create the optimizer and the loss function. Write the training script that will train the model and print the training loss. (20%)
    3. Train three versions of the model, called model2, model4, model8, by only alternating the dimension of the latent representation $r_{dim}$ to be 2, 4, and 8, respectively. Using the provided test_data.csv and the observation pairs provided in test_observations.pt, calculate the average error for each model. Provide results for all these calculations in your report. (30%)
    4.  Given you results, which latent representation ( $r_{dim}$ ) yielded the best results and which latent representation ($r_{dim}$) offered the  most significant improvement? How many independent variables were used to generate the training and test data? Provide explanation. (20%)
    5. For $r_{dim}$ which offered the most improvement, how capable your model was in terms of disentagling the independent variables? Hint: check the correlations between the dimensions of the latent representation. Provide explanation. (10%)